# CortexCode: Brain-Inspired Code Completion on Colab T4

[![GitHub](https://img.shields.io/badge/github-repo-blue)](https://github.com/srivtx/ai-miden/tree/main/cortexcode)

A small (~5M params) code completion model that uses 5 MSPCH (Multi-Scale Predictive Coding Hypothesis) features from the `nature/` curriculum:
1. **Multi-system memory** (slow transformer + fast key-value retrieval)
2. **Replay-driven consolidation** (sleep-driven)
3. **Neuromodulator gating** (DA modulates attention)
4. **Homeostatic plasticity** (Turrigiano scaling)
5. **Variational surprise** (cosine similarity for retrieval)

**Trains on a T4 GPU in 1-2 hours. Runs anywhere: phone, laptop, edge device.**

## Step 1: Verify GPU is enabled

If you haven't already, click `Runtime` → `Change runtime type` → set Hardware accelerator to **T4 GPU** → Save.

Then run this cell:

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. CPU training is 50-100x slower.")
    print("Go to Runtime > Change runtime type > T4 GPU")

## Step 2: Get the code

Clone the repo and cd into the cortexcode directory:

In [ ]:
!git clone https://github.com/srivtx/ai-miden.git
%cd ai-miden/cortexcode
!ls -la

## Step 3: Get a Python codebase to train on

Three options. Pick one.

In [ ]:
# Option A: Use the included gm/ curriculum code as a test set
import shutil, os
if os.path.exists("/content/codebase"):
    shutil.rmtree("/content/codebase")
shutil.copytree("/content/ai-miden/docs/gm", "/content/codebase")
print("Copied gm/ curriculum to /content/codebase")

import os
py_files = []
for root, _, files in os.walk("/content/codebase"):
    for f in files:
        if f.endswith(".py") and "__pycache__" not in root:
            py_files.append(os.path.join(root, f))
print(f"Found {len(py_files)} Python files")
print(f"Total size: {sum(os.path.getsize(f) for f in py_files):,} bytes")

In [ ]:
# Option B: Drag a folder of Python files into the file browser (left sidebar)
# to /content/codebase, then run:
import os
if not os.path.exists("/content/codebase"):
    print("Drag a folder of .py files into /content/codebase using the file browser")
    print("Then re-run this cell")
else:
    py_files = []
    for root, _, files in os.walk("/content/codebase"):
        for f in files:
            if f.endswith(".py") and "__pycache__" not in root:
                py_files.append(os.path.join(root, f))
    print(f"Found {len(py_files)} Python files")
    print(f"Total size: {sum(os.path.getsize(f) for f in py_files):,} bytes")

In [ ]:
# Option C: Clone your own repo (replace with your URL)
# !git clone https://github.com/YOUR_NAME/YOUR_CODE.git /content/codebase

## Step 4: Train

**Recommended settings for T4 free tier (1-2 hours):**

- `--steps 2000` and `--dim 256`: ~30-60 min on T4
- `--steps 5000` and `--dim 384`: ~2-3 hours on T4
- For fast iteration: `--steps 500`

In [ ]:
!python cortexcode_torch.py train \
    --data-dir /content/codebase \
    --steps 2000 \
    --batch-size 16 \
    --block-size 128 \
    --lr 1e-3 \
    --dim 256 \
    --n-layers 4 \
    --n-heads 4 \
    --ffn-dim 512 \
    --out /content/cortexcode.pt

## Step 5: Sample

Once training is done, try prompts from your codebase:

In [ ]:
import os
if not os.path.exists("/content/cortexcode.pt"):
    print("ERROR: /content/cortexcode.pt not found. Run Step 4 (training) first.")
    print("If training failed, check the troubleshooting cell below.")
else:
    !python cortexcode_torch.py sample \
        --prompt "def my_function(" \
        --n-tokens 100 \
        --temperature 0.7 \
        --model /content/cortexcode.pt

In [ ]:
# Try your own prompt
!python cortexcode_torch.py sample \
    --prompt "class MyModel:" \
    --n-tokens 100 \
    --temperature 0.7 \
    --model /content/cortexcode.pt

## Step 6: Save the model to Drive (so it survives Colab restarts)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/cortexcode.pt /content/drive/MyDrive/cortexcode.pt
print("Model saved to Google Drive at MyDrive/cortexcode.pt")

## Troubleshooting

**`No module named cortexcode_torch`**: Make sure you ran the `%cd ai-miden/cortexcode` cell.

**`CUDA out of memory`**: Reduce `--batch-size 8` or `--dim 192`.

**`Found no files at /content/codebase`**: Run Step 3 first.

**Output is gibberish after 30 min**: Train longer (`--steps 5000`) or use a larger codebase.

**Training is slow**: Make sure T4 GPU is enabled (Runtime > Change runtime type).

## What's next

After training, you have a 5M-parameter brain-inspired model that knows your codebase. The model is small enough to run on a phone, free to use, and never sends your code anywhere.

**To use it locally**: download the `.pt` file from Drive, run `cortexcode_torch.py sample --model cortexcode.pt`.

**To use it as a tool**: integrate with VS Code, Neovim, or any editor that can call Python.

**To extend it**: train on more code, increase `--dim`, add more layers, or use the `nature/` curriculum's other principles.

**The research**: see `docs/nature/12_unification/MSPCH.md` for the full thesis and `docs/nature/12_unification/PROPOSAL_molecular_cognitive_coupling.md` for the next research direction.